⏱️ **Time required:** ~3 minutes | **Type:** Domain pipeline (run all cells)

# 🛡️ RideFlow Operations — Domain Pipeline

**Domain Owner:** Trust & Safety Team  
**Systems:** zendesk · checkr · twilio  

This notebook represents the **Operations domain team's** autonomous pipeline. It demonstrates a **multi-system domain** — a single domain team that owns data from 3 different SaaS vendors. Each system has its own registry and contracts, but they share a unified domain governance model.

| System | Bronze Entities | Silver Entities |
| :-- | :-- | :-- |
| **Zendesk** | `support_tickets` | `silver_zendesk_support_tickets` |
| **Checkr** | `background_checks`, `driver_licences` | `silver_checkr_driver_verifications`, `silver_checkr_driver_licences` |
| **Twilio** | `sms_logs` | — |

> **Cross-Domain Dependency:** This pipeline reads `silver_rideflow_driver_profiles` from the Marketplace domain to enrich support tickets and background checks with driver metadata. Run `07b_rideflow_marketplace.ipynb` first.

---
## Step 1 · Environment Setup

In [ ]:
import os
import sys
from pathlib import Path
import polars as pl
import lakelogic as ll

PROJECT_ROOT = Path(".").resolve()
LAKEHOUSE = PROJECT_ROOT / "lakehouse"
ENV = "local"

print(f"Project Root : {PROJECT_ROOT}")
print(f"Lakehouse    : {LAKEHOUSE}")

---
## Step 2 · Load Operations Registries (Multi-System)

The Operations domain owns 3 independent systems. Each gets its own `DomainRegistry`, but they share domain-level config from `_domain.yaml` (SLOs, compliance, observatory).

In [ ]:
from lakelogic.core.registry import DomainRegistry

ops_base = PROJECT_ROOT / "assets" / "domains_rideflow" / "operations"

registries = {}
for system_name in ["zendesk", "checkr", "twilio"]:
    sys_yaml = ops_base / system_name / "_system.yaml"
    registries[system_name] = DomainRegistry.from_yaml(str(sys_yaml))
    contracts = registries[system_name].get_active_contracts()
    print(f"\n📦 {system_name} — {len(contracts)} active contracts")
    for c in contracts:
        print(f"  [{c.target_layer:6s}] {c.entity}")

---
## Step 3 · Generate Synthetic Operations Data

We generate realistic data for all 3 systems, referencing driver IDs from the Marketplace domain to create authentic cross-domain foreign key relationships.

In [ ]:
import random
import string
import json
from datetime import datetime, timezone, timedelta

# Read Marketplace driver IDs and rider IDs for cross-domain references
marketplace_drivers_path = LAKEHOUSE / "marketplace" / "silver" / "silver_rideflow_driver_profiles"
marketplace_trips_path = LAKEHOUSE / "marketplace" / "silver" / "silver_rideflow_trips"

if marketplace_drivers_path.exists():
    drivers_df = pl.read_delta(str(marketplace_drivers_path))
    driver_ids = drivers_df.select("driver_id").unique().to_series().to_list()
    print(f"✅ Loaded {len(driver_ids)} driver IDs from Marketplace domain")
else:
    print("⚠️ Marketplace Silver drivers not found. Using synthetic placeholders.")
    driver_ids = [f"DRV-{i:06d}" for i in range(200)]

if marketplace_trips_path.exists():
    trips_df = pl.read_delta(str(marketplace_trips_path))
    rider_ids = trips_df.select("rider_id").unique().to_series().to_list()
    trip_ids = trips_df.select("trip_id").unique().to_series().to_list()
    print(f"✅ Loaded {len(rider_ids)} rider IDs, {len(trip_ids)} trip IDs")
else:
    rider_ids = [f"RDR-{i:06d}" for i in range(300)]
    trip_ids = [f"TRP-{i:06d}" for i in range(500)]

In [ ]:
def gen_id(prefix, length=8):
    return f"{prefix}_{''.join(random.choices(string.ascii_lowercase + string.digits, k=length))}"


def gen_phone():
    return f"+1{random.randint(200, 999)}{random.randint(1000000, 9999999)}"


def gen_email(name):
    domain = random.choice(["gmail.com", "yahoo.com", "outlook.com", "icloud.com"])
    return f"{name.lower().replace(' ', '.')}{random.randint(1, 99)}@{domain}"


now = datetime.now(timezone.utc)
first_names = ["James", "Maria", "David", "Sarah", "Michael", "Jessica", "Carlos", "Emily", "Ahmed", "Priya"]
last_names = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez"]

# ── Zendesk Support Tickets ──────────────────────────────────────────────
n_tickets = 800
tickets = []
for i in range(n_tickets):
    fname = random.choice(first_names)
    lname = random.choice(last_names)
    name = f"{fname} {lname}"
    created = now - timedelta(hours=random.randint(1, 720))
    solved = created + timedelta(hours=random.randint(1, 48)) if random.random() > 0.3 else None

    subject_templates = [
        f"Trip {random.choice(trip_ids[:100])} — driver was rude",
        "App crashed during ride booking",
        "Incorrect fare charged for my trip",
        "Driver took a longer route",
        "Payment not processed correctly",
        "Account locked — need help",
        "Promo code not applied",
        "Safety concern during ride",
        "Lost item in vehicle",
        "Requesting refund for cancellation fee",
    ]

    tickets.append(
        {
            "ticket_id": gen_id("ZEN"),
            "requester_email": gen_email(name),
            "requester_name": name,
            "requester_phone": gen_phone(),
            "subject": random.choice(subject_templates),
            "ticket_description": f"Rider {random.choice(rider_ids[:50])} reported an issue with trip {random.choice(trip_ids[:100])}",
            "status": random.choices(["new", "open", "pending", "solved", "closed"], weights=[5, 15, 10, 50, 20])[0],
            "priority": random.choices(["low", "normal", "high", "urgent"], weights=[20, 50, 25, 5])[0],
            "channel": random.choices(["email", "chat", "phone", "web"], weights=[30, 35, 15, 20])[0],
            "assignee": random.choice(["Agent_A", "Agent_B", "Agent_C", "Agent_D"]),
            "tags_json": json.dumps(
                random.sample(
                    ["billing", "safety", "app", "driver", "rider", "promo", "refund"], k=random.randint(1, 3)
                )
            ),
            "created_at": created.isoformat(),
            "updated_at": (created + timedelta(hours=random.randint(1, 12))).isoformat(),
            "solved_at": solved.isoformat() if solved else "",
            "satisfaction_rating": str(random.choice(["good", "bad", ""])),
        }
    )

# ── Checkr Background Checks ─────────────────────────────────────────────
n_checks = min(len(driver_ids), 400)
bg_checks = []
for i in range(n_checks):
    driver = driver_ids[i % len(driver_ids)]
    fname = random.choice(first_names)
    lname = random.choice(last_names)
    full_name = f"{fname} {lname}"
    created = now - timedelta(days=random.randint(1, 365))

    bg_checks.append(
        {
            "report_id": gen_id("RPT"),
            "driver_id": driver,
            "full_name": full_name,
            "date_of_birth": f"{random.randint(1970, 2000)}-{random.randint(1, 12):02d}-{random.randint(1, 28):02d}",
            "ssn_last_four": str(random.randint(1000, 9999)),
            "licence_number": f"DL-{random.randint(10000000, 99999999)}",
            "address": f"{random.randint(100, 9999)} {random.choice(['Oak', 'Main', 'Elm', 'Park', 'Cedar'])} {random.choice(['St', 'Ave', 'Blvd', 'Dr'])}",
            "status": random.choices(["clear", "consider", "suspended", "dispute"], weights=[80, 12, 5, 3])[0],
            "result": random.choices(["pass", "review", "fail"], weights=[85, 10, 5])[0],
            "completed_at": (created + timedelta(days=random.randint(1, 7))).isoformat(),
            "created_at": created.isoformat(),
            "adjudication": random.choice(["approved", "pending", ""]),
        }
    )

# ── Checkr Driver Licences ───────────────────────────────────────────────
n_licences = min(len(driver_ids), 350)
licences = []
for i in range(n_licences):
    fname = random.choice(first_names)
    lname = random.choice(last_names)
    licences.append(
        {
            "file_path": f"/data/licences/{driver_ids[i % len(driver_ids)]}.pdf",
            "full_name": f"{fname} {lname}",
            "licence_number": f"DL-{random.randint(10000000, 99999999)}",
            "date_of_birth": f"{random.randint(1970, 2000)}-{random.randint(1, 12):02d}-{random.randint(1, 28):02d}",
            "expiry_date": f"{random.randint(2025, 2030)}-{random.randint(1, 12):02d}-{random.randint(1, 28):02d}",
            "vehicle_classes": random.choice(["B", "B, C", "A, B", "B, C, D"]),
        }
    )

# ── Twilio SMS Logs ──────────────────────────────────────────────────────
n_sms = 1200
sms_logs = []
sms_templates = [
    "Your RideFlow driver {} is arriving in {} min.",
    "Your trip receipt: {}. Total: £{:.2f}",
    "Your verification code is: {}",
    "Your driver {} has been verified.",
    "Rate your trip with driver {}: rideflow.com/rate/{}",
]
for i in range(n_sms):
    sent = now - timedelta(hours=random.randint(1, 168))
    status = random.choices(["delivered", "failed", "undelivered", "sent", "queued"], weights=[82, 5, 3, 8, 2])[0]
    delivered = (sent + timedelta(seconds=random.randint(1, 30))) if status == "delivered" else None

    msg_body = random.choice(sms_templates).format(
        random.choice(driver_ids[:50]),
        random.randint(1, 15),
    )

    sms_logs.append(
        {
            "message_sid": gen_id("SM", 32),
            "recipient_phone": gen_phone(),
            "sender_phone": "+18005551234",
            "message_body": msg_body,
            "status": status,
            "direction": random.choices(["outbound-api", "inbound"], weights=[95, 5])[0],
            "num_segments": str(random.choices([1, 2], weights=[90, 10])[0]),
            "price": f"{random.uniform(0.005, 0.03):.4f}",
            "price_unit": "USD",
            "sent_at": sent.isoformat(),
            "delivered_at": delivered.isoformat() if delivered else "",
            "error_code": str(random.choice([30001, 30003, 30006])) if status in ("failed", "undelivered") else "",
        }
    )

df_tickets = pl.DataFrame(tickets)
df_bg_checks = pl.DataFrame(bg_checks)
df_licences = pl.DataFrame(licences)
df_sms = pl.DataFrame(sms_logs)

print("Generated:")
print(f"  🎫 {len(tickets):,} Zendesk support tickets")
print(f"  🔍 {len(bg_checks):,} Checkr background checks")
print(f"  🪪 {len(licences):,} Checkr driver licences")
print(f"  📱 {len(sms_logs):,} Twilio SMS logs")

---
## Step 4 · Land Synthetic Data

Write each system's data into its own landing zone, simulating real SaaS webhook integrations.

In [ ]:
landing_root = LAKEHOUSE / "_data" / "operations"

datasets = [
    ("zendesk", "support_tickets", df_tickets),
    ("checkr", "background_checks", df_bg_checks),
    ("checkr", "driver_licences", df_licences),
    ("twilio", "sms_logs", df_sms),
]

for system, entity, df in datasets:
    dest = landing_root / system / entity
    dest.mkdir(parents=True, exist_ok=True)
    csv_path = dest / f"{entity}.csv"
    df.write_csv(str(csv_path))
    print(f"  ✅ {system}/{entity}: {len(df):,} rows → {csv_path.relative_to(LAKEHOUSE)}")

---
## Step 5 · Run Pipeline Per System

Each system in the Operations domain runs its own independent pipeline. This is the key multi-system pattern: **one domain owns multiple independent pipelines**, each governed by its own system registry but inheriting domain-level compliance and SLOs.

In [ ]:
from lakelogic.pipeline.runner import LakehousePipeline

summaries = {}
for system_name, registry in registries.items():
    print(f"\n{'=' * 60}")
    print(f"🔧 Running pipeline: operations / {system_name}")
    print(f"{'=' * 60}")

    runner = LakehousePipeline(registry, engine="polars", storage_root=str(LAKEHOUSE))

    summary = runner.run(target_layers="bronze,silver", dry_run=False, environment=ENV)
    summaries[system_name] = summary
    print(summary)

---
## Step 6 · Cross-Domain Enrichment: Tickets × Driver Profiles

This is the Data Mesh moment: the Operations team reads a **published data product** from the Marketplace domain to enrich their support tickets with driver metadata. They never touch Marketplace's internal Bronze tables.

In [ ]:
# Read our Silver tickets
tickets_path = LAKEHOUSE / "operations" / "silver" / "silver_zendesk_support_tickets"

if tickets_path.exists():
    our_tickets = pl.read_delta(str(tickets_path))
else:
    our_tickets = df_tickets  # Fallback to in-memory
    print("⚠️ Using in-memory tickets (Silver not materialised)")

# Read Marketplace data product
driver_profiles_path = LAKEHOUSE / "marketplace" / "silver" / "silver_rideflow_driver_profiles"

if driver_profiles_path.exists():
    marketplace_drivers = pl.read_delta(str(driver_profiles_path))

    # Extract driver IDs mentioned in ticket descriptions
    enriched = (
        our_tickets.with_columns(pl.col("ticket_description").str.extract(r"(DRV-\d+)", 1).alias("mentioned_driver_id"))
        .filter(pl.col("mentioned_driver_id").is_not_null())
        .join(
            marketplace_drivers.select(["driver_id", "city_code"]),
            left_on="mentioned_driver_id",
            right_on="driver_id",
            how="inner",
        )
    )

    print("🔗 Cross-Domain Enrichment")
    print(f"   Total tickets          : {len(our_tickets):,}")
    print(f"   Driver-linked tickets  : {len(enriched):,}")
    if len(enriched) > 0:
        display(enriched.select(["ticket_id", "subject", "priority", "mentioned_driver_id", "city_code"]).head(10))
else:
    print("⚠️ Marketplace driver profiles not found. Cross-domain enrichment skipped.")

---
## Step 7 · Driver Verification Summary

Combine Checkr background check results with driver profiles to produce a trust & safety overview.

In [ ]:
verifications_path = LAKEHOUSE / "operations" / "silver" / "silver_checkr_driver_verifications"

if verifications_path.exists():
    verifications = pl.read_delta(str(verifications_path))
else:
    verifications = df_bg_checks
    print("⚠️ Using in-memory background checks")

status_summary = verifications.group_by("status").agg(pl.count().alias("count")).sort("count", descending=True)

result_summary = verifications.group_by("result").agg(pl.count().alias("count")).sort("count", descending=True)

print("\n📊 Background Check Status Distribution")
display(status_summary)

print("\n📊 Verification Result Distribution")
display(result_summary)

---
## ✅ Data Products Published

The Operations domain pipeline has completed across all 3 systems. Published data products:

| Data Product | System | Consumers |
| :-- | :-- | :-- |
| `silver_zendesk_support_tickets` | Zendesk | CSAT analytics, escalation workflows |
| `silver_checkr_driver_verifications` | Checkr | Trust scoring, driver onboarding |
| `silver_checkr_driver_licences` | Checkr | Regulatory compliance |
| `bronze_twilio_sms_logs` | Twilio | Communication audit trail |

### Multi-System Architecture

```
Operations Domain (Trust & Safety)
┌───────────────────────────────────────────────┐
│                                               │
│  ┌──────────┐  ┌─────────┐  ┌──────────┐    │
│  │ Zendesk  │  │ Checkr  │  │  Twilio  │    │
│  │ (tickets)│  │ (bgchk) │  │  (sms)   │    │
│  └────┬─────┘  └────┬────┘  └────┬─────┘    │
│       │              │            │           │
│  Bronze → Silver  Bronze → Silver  Bronze     │
│                                               │
│  Shared: _domain.yaml (SLO, compliance, cost) │
└───────────────────────────────────────────────┘
        │                    │
        ▼                    ▼
  Marketplace          Marketplace
  driver_profiles      trips
  (data product)       (data product)
```

### Next Notebooks

- **`07e_rideflow_marketing.ipynb`** — Marketing domain (GA, HubSpot, Meta, Google Ads)
- **`07g_compliance_gdpr_rtbf.ipynb`** — Cross-domain privacy erasure
- **`07i_data_mesh_dashboards.ipynb`** — Unified mesh observability